In [ ]:
import polars as pl
dataset = pl.read_csv('../data/processed/dataset_clean_onehot.csv')


In [ ]:
pl.Config.set_tbl_rows(-1)
coocurences = dataset\
    .with_columns(pl.col('StimSite').str.strip_chars_end('_123456789'))\
    .group_by(['Error_type_annot', 'StimSite']).len()\
    .sort(['StimSite', 'len'], descending=True)\
    .filter(pl.col('Error_type_annot') != 'нет')
#    .group_by(['StimSite']).head(4)\

In [ ]:
coocurences.sort(['len'], descending=True)\
            .filter(pl.col('Error_type_annot') != 'задержка')\
            .with_columns(100 * pl.col('len') / \
                         pl.sum('len').over('Error_type_annot'))\
            .sort('Error_type_annot')


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
from numpy import newaxis

pivot = coocurences.pivot(index='Error_type_annot',
                          on='StimSite',
                          values='len')\
                    .fill_null(0)

rows = pivot['Error_type_annot'].to_list()
cols = pivot.columns[1:]

pivot_normalized = pivot[:, 1:].to_numpy()
pivot_normalized = pivot_normalized / \
    pivot_normalized.sum(axis=1)[:, newaxis]

pivot_normalized_2 = pivot[:, 1:].to_numpy()
pivot_normalized_2 = pivot_normalized_2 / \
    pivot_normalized_2.sum(axis=0)[newaxis, :]

plt.imshow((pivot_normalized), cmap='gray', aspect='auto')
plt.xticks(range(len(cols)), cols, rotation=90)
plt.yticks(range(len(rows)), rows)
plt.plot()


In [ ]:
plt.imshow((pivot_normalized_2.T), cmap='gray', aspect='auto')
plt.xticks(range(len(rows)), rows, rotation=90)
plt.yticks(range(len(cols)), cols)
plt.plot()
